In [ ]:
import pandas as pd
import numpy as np


corr = pd.read_csv("results/correlation_matrix.csv", index_col=0)
vif  = pd.read_csv("results/vif_results.csv")

BASE_FEATURES_30 = [
    "PIGD", "UPO_part2_tot", "UPO_part3_tot", "UPO27", "UPO_part4_tot",
    "STROOPW", "MMSE_tot", "IQCODE_mean", "madrs10", "madrs_tot", "npi8hi",
    "pdss02", "pdss_tot", "epwo6", "URINDYS", "CONSTIPAT", "PressStand_Dia",
    "fss2", "fss5", "sf36pcs", "PHYFUN", "sf36pf", "fss_sum", "SEON",
    "UPO15", "HYON", "MCI_15SD_MDScriteria", "STROOPC",
]

vif_dict = dict(zip(vif["Feature"], vif["VIF"]))

def get_problematic_pairs(features, threshold):
    pairs = []
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            f1, f2 = features[i], features[j]
            if f1 in corr.index and f2 in corr.index:
                val = abs(corr.loc[f1, f2])
                if val >= threshold:
                    pairs.append((f1, f2, round(val, 3)))
    return sorted(pairs, key=lambda x: -x[2])

def greedy_remove(features, threshold):
    remaining = features.copy()
    removed   = []
    changed   = True
    while changed:
        changed = False
        pairs   = get_problematic_pairs(remaining, threshold)
        if not pairs:
            break
        problematic = {}
        for f1, f2, val in pairs:
            problematic[f1] = max(problematic.get(f1, 0), vif_dict.get(f1, 0))
            problematic[f2] = max(problematic.get(f2, 0), vif_dict.get(f2, 0))
        to_remove = max(problematic, key=problematic.get)
        remaining.remove(to_remove)
        removed.append((to_remove, round(vif_dict.get(to_remove, 0), 1)))
        changed = True
    return remaining, removed

# ── Kjør for ulike terskler ───────────────────────────────────────────────────
print("=== N=20 (threshold r < 0.78) ===")
feat_20, removed_20 = greedy_remove(BASE_FEATURES_30, 0.78)
print(f"Features ({len(feat_20)}): {feat_20}")
print(f"Removed:  {[r[0] for r in removed_20]}\n")

print("=== N=15 (threshold r < 0.65) ===")
feat_15, removed_15 = greedy_remove(BASE_FEATURES_30, 0.65)
print(f"Features ({len(feat_15)}): {feat_15}")
print(f"Removed:  {[r[0] for r in removed_15]}\n")

print("=== N=10 (lowest VIF from N=15) ===")
vif_15 = [(f, vif_dict.get(f, 0)) for f in feat_15]
vif_15_sorted = sorted(vif_15, key=lambda x: x[1])
feat_10 = [f for f, v in vif_15_sorted[:10]]
print(f"Features ({len(feat_10)}): {feat_10}")

=== N=20 (threshold r < 0.78) ===
Features (20): ['PIGD', 'UPO_part2_tot', 'UPO_part3_tot', 'UPO27', 'UPO_part4_tot', 'STROOPW', 'MMSE_tot', 'IQCODE_mean', 'madrs10', 'madrs_tot', 'npi8hi', 'pdss02', 'pdss_tot', 'epwo6', 'URINDYS', 'CONSTIPAT', 'PressStand_Dia', 'fss2', 'fss5', 'sf36pcs']
Removed:  ['PHYFUN', 'sf36pf', 'fss_sum', 'SEON', 'UPO15', 'HYON', 'MCI_15SD_MDScriteria', 'STROOPC']

=== N=15 (threshold r < 0.65) ===
Features (15): ['PIGD', 'UPO_part2_tot', 'UPO_part4_tot', 'IQCODE_mean', 'madrs10', 'madrs_tot', 'npi8hi', 'pdss02', 'pdss_tot', 'epwo6', 'URINDYS', 'CONSTIPAT', 'PressStand_Dia', 'fss2', 'sf36pcs']
Removed:  ['PHYFUN', 'sf36pf', 'fss_sum', 'SEON', 'fss5', 'UPO15', 'HYON', 'UPO27', 'MCI_15SD_MDScriteria', 'UPO_part3_tot', 'STROOPC', 'STROOPW', 'MMSE_tot']

=== N=10 (lowest VIF from N=15) ===
Features (10): ['npi8hi', 'PressStand_Dia', 'URINDYS', 'CONSTIPAT', 'pdss02', 'epwo6', 'UPO_part4_tot', 'madrs10', 'PIGD', 'pdss_tot']
